In [1]:
"""
Notebook script (.py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación y combinado.

Compatible con pandas >= 2.0
"""

import os
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
import joblib

# ---------------------- CONFIGURACION ----------------------

FILE_PATHS = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

OUTPUT_DIR = Path(r"~/Documents/GitHub/TFGFinal/Datos TFG/DatasetsFinales").expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72

# Estrategias de imputación para columnas específicas (solo las que existen en los CSV)
IMPUTE_STRATEGIES = {
    'NO': 'time_interpolate',
    'NO2': 'time_interpolate',
    'NOx': 'time_interpolate',
    'O3': 'time_interpolate',
    'Veloc.': 'time_interpolate',
    'Direc.': 'circular',          # Se transformará a seno/coseno y luego se imputarán esas componentes
    'Temp.': 'time_interpolate',
    'R.Sol.': 'time_interpolate',
    'Estacion': 'ffill',
    'Transecto': 'ffill',
    'Dist.': 'ffill',
    'Angulo': 'circular'
}

BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']   # Predictores numéricos base
CATEGORICAL = ['Estacion', 'Transecto']                              # Categóricas para one-hot
CIRCULAR_ORIG = ['Direc.', 'Angulo']                                 # Variables angulares (se transforman)
TARGET = 'O3'                                                         # Variable objetivo

# ---------------------- FUNCIONES DE PREPROCESADO ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    """Convierte una columna a datetime, la establece como índice y completa la serie horaria."""
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df = df.dropna(subset=[col])
        df = df.set_index(col)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No hay índice datetime válido")
    df = df.sort_index()
    # Eliminar duplicados temporales
    if not df.index.is_unique:
        df = df[~df.index.duplicated(keep='first')]
    # Crear índice horario completo
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    df = df.reindex(full_idx)
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    """Añade características cíclicas (seno/coseno) de hora, día, semana, mes y año."""
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

    years = year - year.min()
    # Evitar división por cero si solo hay un año
    denom = max(year.max() - year.min() + 1, 1)
    d['year_sin'] = np.sin(2 * np.pi * years / denom)
    d['year_cos'] = np.cos(2 * np.pi * years / denom)
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    """Convierte una serie de ángulos (grados) a seno y coseno."""
    radians = np.deg2rad(series)
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(radians),
        f'{prefix}_cos': np.cos(radians)
    }, index=series.index)


def impute_variable(df: pd.DataFrame, col: str, strategy: str) -> pd.Series:
    """
    Imputa una columna según la estrategia indicada.
    Estrategias: 'time_interpolate', 'ffill', 'median', 'constant'.
    """
    s = df[col]
    if strategy == 'time_interpolate':
        s_imputed = s.interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strategy == 'ffill':
        return s.ffill().bfill()
    elif strategy == 'median':
        return s.fillna(s.median())
    elif strategy == 'constant':
        return s.fillna(0)
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")


def preprocess_single_csv(path: str) -> pd.DataFrame:
    """Carga, limpia e imputa un único archivo CSV."""
    print(f"  Preprocesando {Path(path).name}...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # 1. Índice temporal completo
    df = ensure_datetime_index(df, 'datetime')

    # 2. Añadir características temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # 3. Transformar variables circulares a seno/coseno y eliminar las originales
    for circ in CIRCULAR_ORIG:
        if circ in df.columns:
            sincos = circular_to_sincos(df[circ], circ.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)
            df.drop(columns=[circ], inplace=True)  # Ya no necesitamos la original

    # 4. Definir estrategias de imputación para todas las columnas presentes
    #    (se prioriza lo definido en IMPUTE_STRATEGIES, por defecto 'time_interpolate' para numéricas)
    impute_map = {}
    for col in df.columns:
        if col in IMPUTE_STRATEGIES:
            strat = IMPUTE_STRATEGIES[col]
            # Si la estrategia es 'circular', ya hemos creado las componentes, así que las tratamos como 'time_interpolate'
            if strat == 'circular':
                strat = 'time_interpolate'
            impute_map[col] = strat
        elif col in BASE_FEATURES or col == TARGET or col.endswith('_sin') or col.endswith('_cos'):
            impute_map[col] = 'time_interpolate'
        elif col in CATEGORICAL:
            impute_map[col] = 'ffill'
        else:
            # Para cualquier otra columna (p.ej., 'Dist.') usar la estrategia definida o 'median'
            impute_map[col] = IMPUTE_STRATEGIES.get(col, 'median')

    # 5. Aplicar imputación columna por columna
    for col, strat in impute_map.items():
        if col in df.columns:
            df[col] = impute_variable(df, col, strat)
        else:
            # Esto no debería ocurrir porque iteramos sobre las columnas existentes
            pass

    # 6. Garantizar que las columnas base y el target no tengan NaNs (relleno final con mediana)
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return df


# ---------------------- VENTANAS DESLIZANTES ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: Optional[List[str]] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Genera ventanas deslizantes de entrada-salida.

    - Si model_type es 'rf' o 'xgb': aplana las ventanas y devuelve DataFrames (2D).
    - Si model_type es 'rnn' o 'lstm': devuelve arrays 3D (muestras, tiempo, características).

    features: lista de nombres de columnas a usar como predictores.
              Si es None, se toman todas las columnas numéricas del dataframe.
    """
    if features is None:
        # Por defecto, usar todas las columnas numéricas (excluyendo el target si está)
        features = df.select_dtypes(include=[np.number]).columns.tolist()
        if target_col in features:
            features.remove(target_col)

    # Asegurar que todas las features existen
    features = [f for f in features if f in df.columns]

    X_list, y_list, idxs = [], [], []
    total_len = len(df)

    for start in range(total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours

        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values

        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue

        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])   # timestamp del último instante de entrada

    if model_type in ['rf', 'xgb']:
        # Aplanar: de (n_muestras, tiempo, n_feat) a (n_muestras, tiempo * n_feat)
        X_flat = np.array(X_list)                     # (n, t, f)
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)                       # (n, output_hours)

        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d, index=idxs)
        y_df = pd.DataFrame(y_2d, columns=y_cols, index=idxs)
        return X_df, y_df
    else:
        # Para RNN/LSTM se mantiene la estructura 3D
        return np.array(X_list), np.array(y_list)


# ---------------------- FUNCIÓN PRINCIPAL ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    """
    Procesa todos los archivos, guarda datasets individuales y combinados.
    """
    # Acumuladores para los datasets combinados
    all_rf_X = []
    all_rf_y = []
    all_xgb_X = []
    all_xgb_y = []
    all_rnn_X = []
    all_rnn_y = []

    for p in file_paths:
        p = str(Path(p).expanduser())
        station_name = Path(p).stem
        print(f"\nProcesando estación: {station_name}")

        # Preprocesar el CSV
        df = preprocess_single_csv(p)

        # ------------------------------------------------------------
        # 1. Datasets para RF y XGB (con one-hot encoding de categóricas)
        # ------------------------------------------------------------
        # Identificar columnas categóricas que existen
        cat_cols = [c for c in CATEGORICAL if c in df.columns]
        df_tab = pd.get_dummies(df, columns=cat_cols)   # one-hot

        # Para RF/XGB queremos todas las columnas numéricas, incluyendo las dummies
        feature_cols_tab = df_tab.select_dtypes(include=[np.number]).columns.tolist()
        if TARGET in feature_cols_tab:
            feature_cols_tab.remove(TARGET)

        # Generar ventanas para RF
        X_rf, y_rf = make_sliding_windows(df_tab,
                                          features=feature_cols_tab,
                                          target_col=TARGET,
                                          model_type='rf')
        # Guardar individualmente
        station_dir = output_dir / 'per_station' / station_name
        station_dir.mkdir(parents=True, exist_ok=True)
        X_rf.to_csv(station_dir / 'rf_X.csv')
        y_rf.to_csv(station_dir / 'rf_y.csv')
        # Acumular para combinado
        all_rf_X.append(X_rf)
        all_rf_y.append(y_rf)

        # Generar ventanas para XGB (mismas características que RF)
        X_xgb, y_xgb = make_sliding_windows(df_tab,
                                            features=feature_cols_tab,
                                            target_col=TARGET,
                                            model_type='xgb')
        X_xgb.to_csv(station_dir / 'xgb_X.csv')
        y_xgb.to_csv(station_dir / 'xgb_y.csv')
        all_xgb_X.append(X_xgb)
        all_xgb_y.append(y_xgb)

        # ------------------------------------------------------------
        # 2. Datasets para RNN/LSTM (sin one-hot, solo numéricas continuas)
        # ------------------------------------------------------------
        # Para RNN usamos todas las columnas numéricas (incluyendo sen/cos) pero sin las dummies
        # (las categóricas originales ya no están porque no se han transformado)
        feature_cols_rnn = df.select_dtypes(include=[np.number]).columns.tolist()
        if TARGET in feature_cols_rnn:
            feature_cols_rnn.remove(TARGET)

        X_rnn, y_rnn = make_sliding_windows(df,
                                            features=feature_cols_rnn,
                                            target_col=TARGET,
                                            model_type='rnn')
        # Guardar en un único archivo (sirve tanto para RNN como para LSTM)
        np.savez_compressed(station_dir / 'rnn_lstm_data.npz', X=X_rnn, y=y_rnn)
        all_rnn_X.append(X_rnn)
        all_rnn_y.append(y_rnn)

        print(f"  Ventanas generadas: RF={len(X_rf)}, XGB={len(X_xgb)}, RNN={len(X_rnn)}")

    # ------------------------------------------------------------
    # 3. Datasets combinados (todas las estaciones juntas)
    # ------------------------------------------------------------
    print("\nGenerando datasets combinados...")
    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    # Combinar RF
    if all_rf_X:
        combined_rf_X = pd.concat(all_rf_X, axis=0)
        combined_rf_y = pd.concat(all_rf_y, axis=0)
        combined_rf_X.to_csv(combined_dir / 'rf_X.csv')
        combined_rf_y.to_csv(combined_dir / 'rf_y.csv')

    # Combinar XGB
    if all_xgb_X:
        combined_xgb_X = pd.concat(all_xgb_X, axis=0)
        combined_xgb_y = pd.concat(all_xgb_y, axis=0)
        combined_xgb_X.to_csv(combined_dir / 'xgb_X.csv')
        combined_xgb_y.to_csv(combined_dir / 'xgb_y.csv')

    # Combinar RNN/LSTM (concatenar arrays en el eje 0)
    if all_rnn_X:
        combined_rnn_X = np.concatenate(all_rnn_X, axis=0)
        combined_rnn_y = np.concatenate(all_rnn_y, axis=0)
        np.savez_compressed(combined_dir / 'rnn_lstm_data.npz',
                            X=combined_rnn_X, y=combined_rnn_y)

    print(f"\n¡Proceso completado! Datos guardados en: {output_dir}")


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)


Procesando estación: T1_E1_Alicante
  Preprocesando T1_E1_Alicante.csv...


KeyboardInterrupt: 

In [ ]:
"""
Script para entrenar y evaluar los 4 modelos (RF, XGB, RNN, LSTM)
utilizando los datasets generados por 'generar_datasets.py'.

Se asume que los datos se encuentran en la estructura de carpetas:
  ~/Documents/GitHub/TFGFinal/Datos TFG/DatasetsFinales/
    per_station/
        <nombre_estacion>/
            rf_X.csv, rf_y.csv, xgb_X.csv, xgb_y.csv, rnn_lstm_data.npz
    combined/
            rf_X.csv, rf_y.csv, xgb_X.csv, xgb_y.csv, rnn_lstm_data.npz

El script permite elegir:
  - Modo 'station': procesa una estación concreta (por nombre).
  - Modo 'combined': procesa el dataset combinado.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, Optional, Union

# Modelos de sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# TensorFlow / Keras para RNN y LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuración de rutas
BASE_DIR = Path(r"~/Documents/GitHub/TFGFinal/Datos TFG/DatasetsFinales").expanduser()
OUTPUT_DIR = BASE_DIR / "resultados_entrenamiento"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parámetros globales
INPUT_HOURS = 72
OUTPUT_HOURS = 72
RANDOM_STATE = 42
TEST_SIZE = 0.15          # 15% para test
VAL_SIZE = 0.15           # 15% para validación (sobre el resto después de test)
SCALE_FEATURES = True      # Escalar características (importante para RNN/LSTM y puede ayudar a RF/XGB)

# Selección de modo y estación (cambiar según necesidad)
MODE = 'combined'          # 'station' o 'combined'
STATION_NAME = 'T1_E1_Alicante'   # Solo si MODE == 'station'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def load_rf_xgb_data(mode: str, station: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Carga los datos de RF o XGB (formato CSV con features aplanadas)."""
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    X = pd.read_csv(data_dir / 'rf_X.csv', index_col=0, parse_dates=True)
    y = pd.read_csv(data_dir / 'rf_y.csv', index_col=0, parse_dates=True)
    return X, y

def load_rnn_lstm_data(mode: str, station: Optional[str] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Carga los datos de RNN/LSTM (formato npz con arrays 3D)."""
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    data = np.load(data_dir / 'rnn_lstm_data.npz')
    return data['X'], data['y']

def temporal_train_val_test_split(X, y, val_size=0.15, test_size=0.15, random_state=None):
    """
    Divide los datos en train/val/test respetando el orden temporal.
    Asume que X e y están indexados por tiempo (para pandas DataFrame) o
    que las muestras están en orden cronológico (para arrays numpy).
    """
    n = len(X)
    test_cut = int(n * (1 - test_size))
    val_cut = int(test_cut * (1 - val_size))

    if isinstance(X, pd.DataFrame):
        # Para pandas, mantener índices
        X_train, y_train = X.iloc[:val_cut], y.iloc[:val_cut]
        X_val, y_val = X.iloc[val_cut:test_cut], y.iloc[val_cut:test_cut]
        X_test, y_test = X.iloc[test_cut:], y.iloc[test_cut:]
    else:
        # Para numpy arrays
        X_train, y_train = X[:val_cut], y[:val_cut]
        X_val, y_val = X[val_cut:test_cut], y[val_cut:test_cut]
        X_test, y_test = X[test_cut:], y[test_cut:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def scale_features(X_train, X_val, X_test, feature_names=None):
    """
    Escala las características usando StandardScaler.
    Devuelve los conjuntos escalados y el scaler ajustado.
    """
    scaler = StandardScaler()
    # Ajustar con entrenamiento
    if isinstance(X_train, pd.DataFrame):
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        # Convertir de vuelta a DataFrame si se desea (opcional)
        X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
        X_val_scaled = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
        X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
    else:
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

def evaluate_model(y_true, y_pred, model_name, dataset_type='test'):
    """Calcula métricas y las imprime. Devuelve diccionario con las métricas."""
    mae = mean_absolute_error(y_true, y_pred, multioutput='uniform_average')
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='uniform_average'))
    r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
    print(f"{model_name} - {dataset_type} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

def plot_predictions(y_true, y_pred, model_name, num_samples=5):
    """Dibuja las primeras num_samples predicciones vs reales (para una hora específica, p.ej. horizonte 24)."""
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))
    if num_samples == 1:
        axes = [axes]
    for i in range(num_samples):
        ax = axes[i]
        ax.plot(y_true[i], label='Real', marker='o', linestyle='-')
        ax.plot(y_pred[i], label='Predicho', marker='x', linestyle='--')
        ax.set_title(f'{model_name} - Muestra {i+1}')
        ax.set_xlabel('Horas predicción')
        ax.set_ylabel('O3')
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_predicciones_ejemplo.png')
    plt.show()

# ---------------------- ENTRENAMIENTO DE MODELOS ----------------------

def train_random_forest(X_train, y_train, X_val, y_val, X_test, y_test):
    """Entrena Random Forest multi-output y evalúa."""
    print("\n--- Entrenando Random Forest ---")
    # Hiperparámetros básicos (se pueden ajustar después)
    model = RandomForestRegressor(n_estimators=100, max_depth=20,
                                  random_state=RANDOM_STATE, n_jobs=-1)
    # RandomForestRegressor soporta múltiples salidas directamente
    model.fit(X_train, y_train)

    # Predicciones
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    # Evaluación
    metrics_train = evaluate_model(y_train.values, y_pred_train, "RF", "train")
    metrics_val = evaluate_model(y_val.values, y_pred_val, "RF", "val")
    metrics_test = evaluate_model(y_test.values, y_pred_test, "RF", "test")

    # Guardar modelo
    import joblib
    joblib.dump(model, OUTPUT_DIR / 'random_forest_model.pkl')
    print("Modelo RF guardado en:", OUTPUT_DIR / 'random_forest_model.pkl')

    # Ejemplo de predicciones
    plot_predictions(y_test.values[:5], y_pred_test[:5], "Random Forest")
    return model, metrics_test

def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test):
    """Entrena XGBoost multi-output y evalúa."""
    print("\n--- Entrenando XGBoost ---")
    # XGBRegressor también soporta múltiples salidas
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100,
                             max_depth=6, learning_rate=0.1,
                             random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train.values, y_pred_train, "XGB", "train")
    metrics_val = evaluate_model(y_val.values, y_pred_val, "XGB", "val")
    metrics_test = evaluate_model(y_test.values, y_pred_test, "XGB", "test")

    import joblib
    joblib.dump(model, OUTPUT_DIR / 'xgboost_model.pkl')
    print("Modelo XGB guardado en:", OUTPUT_DIR / 'xgboost_model.pkl')

    plot_predictions(y_test.values[:5], y_pred_test[:5], "XGBoost")
    return model, metrics_test

def build_rnn_model(input_shape, output_steps):
    """Construye un modelo RNN (SimpleRNN) para secuencia a secuencia."""
    model = Sequential([
        SimpleRNN(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        SimpleRNN(64, return_sequences=False),
        Dropout(0.2),
        Dense(output_steps)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(input_shape, output_steps):
    """Construye un modelo LSTM para secuencia a secuencia."""
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(output_steps)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def train_rnn_lstm(model_type, X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Entrena modelo RNN o LSTM.
    model_type: 'rnn' o 'lstm'
    Los datos X son arrays 3D: (muestras, pasos_tiempo, características)
    """
    print(f"\n--- Entrenando {model_type.upper()} ---")
    input_shape = (X_train.shape[1], X_train.shape[2])  # (timesteps, features)
    output_steps = y_train.shape[1]

    if model_type == 'rnn':
        model = build_rnn_model(input_shape, output_steps)
    else:
        model = build_lstm_model(input_shape, output_steps)

    model.summary()

    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    # Predicciones
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train, y_pred_train, model_type.upper(), "train")
    metrics_val = evaluate_model(y_val, y_pred_val, model_type.upper(), "val")
    metrics_test = evaluate_model(y_test, y_pred_test, model_type.upper(), "test")

    # Guardar modelo y historial
    model.save(OUTPUT_DIR / f'{model_type}_model.h5')
    np.save(OUTPUT_DIR / f'{model_type}_history.npy', history.history)
    print(f"Modelo {model_type.upper()} guardado en:", OUTPUT_DIR / f'{model_type}_model.h5')

    # Gráficas de entrenamiento
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'{model_type.upper()} - Loss')
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history.history['mae'], label='train_mae')
    plt.plot(history.history['val_mae'], label='val_mae')
    plt.title(f'{model_type.upper()} - MAE')
    plt.legend()
    plt.savefig(OUTPUT_DIR / f'{model_type}_training_history.png')
    plt.show()

    # Ejemplo de predicciones
    plot_predictions(y_test[:5], y_pred_test[:5], model_type.upper())

    return model, metrics_test

# ---------------------- MAIN ----------------------

def main():
    print(f"Modo seleccionado: {MODE}")
    if MODE == 'station':
        print(f"Estación: {STATION_NAME}")

    # 1. Cargar datos para RF/XGB
    X_rf, y_rf = load_rf_xgb_data(MODE, STATION_NAME if MODE=='station' else None)
    print(f"Datos RF/XGB cargados: {X_rf.shape[0]} muestras, {X_rf.shape[1]} features (aplanadas)")

    # División temporal para RF/XGB
    (X_rf_train, y_rf_train), (X_rf_val, y_rf_val), (X_rf_test, y_rf_test) = \
        temporal_train_val_test_split(X_rf, y_rf, val_size=VAL_SIZE, test_size=TEST_SIZE)
    print(f"RF/XGB - Train: {len(X_rf_train)}, Val: {len(X_rf_val)}, Test: {len(X_rf_test)}")

    # Escalado opcional para RF/XGB
    if SCALE_FEATURES:
        X_rf_train, X_rf_val, X_rf_test, scaler_rf = scale_features(
            X_rf_train, X_rf_val, X_rf_test)
        # Guardar scaler por si se necesita para inferencia
        import joblib
        joblib.dump(scaler_rf, OUTPUT_DIR / 'scaler_rf_xgb.pkl')
        print("Scaler para RF/XGB guardado.")

    # 2. Entrenar RF
    rf_model, rf_metrics = train_random_forest(
        X_rf_train, y_rf_train,
        X_rf_val, y_rf_val,
        X_rf_test, y_rf_test
    )

    # 3. Entrenar XGBoost
    xgb_model, xgb_metrics = train_xgboost(
        X_rf_train, y_rf_train,
        X_rf_val, y_rf_val,
        X_rf_test, y_rf_test
    )

    # 4. Cargar datos para RNN/LSTM
    X_rnn, y_rnn = load_rnn_lstm_data(MODE, STATION_NAME if MODE=='station' else None)
    print(f"Datos RNN/LSTM cargados: {X_rnn.shape[0]} muestras, timesteps={X_rnn.shape[1]}, features={X_rnn.shape[2]}")

    # División temporal para RNN/LSTM
    (X_rnn_train, y_rnn_train), (X_rnn_val, y_rnn_val), (X_rnn_test, y_rnn_test) = \
        temporal_train_val_test_split(X_rnn, y_rnn, val_size=VAL_SIZE, test_size=TEST_SIZE)
    print(f"RNN/LSTM - Train: {len(X_rnn_train)}, Val: {len(X_rnn_val)}, Test: {len(X_rnn_test)}")

    # Escalado para RNN/LSTM (importante: escalar características, no la dimensión temporal)
    if SCALE_FEATURES:
        # Reshape a 2D para escalar: (n_samples * timesteps, n_features)
        original_shape = X_rnn_train.shape
        n_samples_train, n_timesteps, n_features = original_shape
        X_rnn_train_2d = X_rnn_train.reshape(-1, n_features)
        X_rnn_val_2d = X_rnn_val.reshape(-1, n_features)
        X_rnn_test_2d = X_rnn_test.reshape(-1, n_features)

        scaler_rnn = StandardScaler()
        X_rnn_train_2d = scaler_rnn.fit_transform(X_rnn_train_2d)
        X_rnn_val_2d = scaler_rnn.transform(X_rnn_val_2d)
        X_rnn_test_2d = scaler_rnn.transform(X_rnn_test_2d)

        # Reconstruir 3D
        X_rnn_train = X_rnn_train_2d.reshape(original_shape)
        X_rnn_val = X_rnn_val_2d.reshape(X_rnn_val.shape)
        X_rnn_test = X_rnn_test_2d.reshape(X_rnn_test.shape)

        # Guardar scaler
        import joblib
        joblib.dump(scaler_rnn, OUTPUT_DIR / 'scaler_rnn_lstm.pkl')
        print("Scaler para RNN/LSTM guardado.")

    # 5. Entrenar RNN
    rnn_model, rnn_metrics = train_rnn_lstm(
        'rnn',
        X_rnn_train, y_rnn_train,
        X_rnn_val, y_rnn_val,
        X_rnn_test, y_rnn_test
    )

    # 6. Entrenar LSTM
    lstm_model, lstm_metrics = train_rnn_lstm(
        'lstm',
        X_rnn_train, y_rnn_train,
        X_rnn_val, y_rnn_val,
        X_rnn_test, y_rnn_test
    )

    # 7. Comparativa final
    print("\n" + "="*50)
    print("RESUMEN DE MÉTRICAS EN TEST")
    print("="*50)
    comparativa = pd.DataFrame({
        'RF': rf_metrics,
        'XGB': xgb_metrics,
        'RNN': rnn_metrics,
        'LSTM': lstm_metrics
    }).T
    print(comparativa)
    comparativa.to_csv(OUTPUT_DIR / 'comparativa_metricas.csv')

    # Guardar las predicciones de test para cada modelo (opcional)
    # ...

    print(f"\nTodos los resultados guardados en: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()

# CON .parquet

In [ ]:
"""
Notebook script (.py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación y combinado.
Los datasets para RF y XGB se guardan en formato Parquet (menor tamaño).
"""

import os
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
import joblib

# ---------------------- CONFIGURACION ----------------------

FILE_PATHS = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

OUTPUT_DIR = Path(r"/Volumes/copia seguridad1/Datos TFG").expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72

# Estrategias de imputación para columnas específicas (solo las que existen en los CSV)
IMPUTE_STRATEGIES = {
    'NO': 'time_interpolate',
    'NO2': 'time_interpolate',
    'NOx': 'time_interpolate',
    'O3': 'time_interpolate',
    'Veloc.': 'time_interpolate',
    'Direc.': 'circular',          # Se transformará a seno/coseno
    'Temp.': 'time_interpolate',
    'R.Sol.': 'time_interpolate',
    'Estacion': 'ffill',
    'Transecto': 'ffill',
    'Dist.': 'ffill',
    'Angulo': 'circular'
}

BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']   # Predictores numéricos base
CATEGORICAL = ['Estacion', 'Transecto']                              # Categóricas para one-hot
CIRCULAR_ORIG = ['Direc.', 'Angulo']                                 # Variables angulares (se transforman)
TARGET = 'O3'                                                         # Variable objetivo

# ---------------------- FUNCIONES DE PREPROCESADO ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    """Convierte una columna a datetime, la establece como índice y completa la serie horaria."""
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df = df.dropna(subset=[col])
        df = df.set_index(col)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No hay índice datetime válido")
    df = df.sort_index()
    # Eliminar duplicados temporales
    if not df.index.is_unique:
        df = df[~df.index.duplicated(keep='first')]
    # Crear índice horario completo
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    df = df.reindex(full_idx)
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    """Añade características cíclicas (seno/coseno) de hora, día, semana, mes y año."""
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

    years = year - year.min()
    denom = max(year.max() - year.min() + 1, 1)
    d['year_sin'] = np.sin(2 * np.pi * years / denom)
    d['year_cos'] = np.cos(2 * np.pi * years / denom)
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    """Convierte una serie de ángulos (grados) a seno y coseno."""
    radians = np.deg2rad(series)
    return pd.DataFrame({
        f'{prefix}_sin': np.sin(radians),
        f'{prefix}_cos': np.cos(radians)
    }, index=series.index)


def impute_variable(df: pd.DataFrame, col: str, strategy: str) -> pd.Series:
    """
    Imputa una columna según la estrategia indicada.
    Estrategias: 'time_interpolate', 'ffill', 'median', 'constant'.
    """
    s = df[col]
    if strategy == 'time_interpolate':
        s_imputed = s.interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strategy == 'ffill':
        return s.ffill().bfill()
    elif strategy == 'median':
        return s.fillna(s.median())
    elif strategy == 'constant':
        return s.fillna(0)
    else:
        raise ValueError(f"Estrategia desconocida: {strategy}")


def preprocess_single_csv(path: str) -> pd.DataFrame:
    """Carga, limpia e imputa un único archivo CSV."""
    print(f"  Preprocesando {Path(path).name}...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # 1. Índice temporal completo
    df = ensure_datetime_index(df, 'datetime')

    # 2. Añadir características temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # 3. Transformar variables circulares a seno/coseno y eliminar las originales
    for circ in CIRCULAR_ORIG:
        if circ in df.columns:
            sincos = circular_to_sincos(df[circ], circ.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)
            df.drop(columns=[circ], inplace=True)

    # 4. Definir estrategias de imputación para todas las columnas presentes
    impute_map = {}
    for col in df.columns:
        if col in IMPUTE_STRATEGIES:
            strat = IMPUTE_STRATEGIES[col]
            if strat == 'circular':
                strat = 'time_interpolate'   # ya tenemos componentes _sin/_cos
            impute_map[col] = strat
        elif col in BASE_FEATURES or col == TARGET or col.endswith('_sin') or col.endswith('_cos'):
            impute_map[col] = 'time_interpolate'
        elif col in CATEGORICAL:
            impute_map[col] = 'ffill'
        else:
            impute_map[col] = IMPUTE_STRATEGIES.get(col, 'median')

    # 5. Aplicar imputación
    for col, strat in impute_map.items():
        if col in df.columns:
            df[col] = impute_variable(df, col, strat)

    # 6. Garantizar que columnas base y target no tengan NaNs
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return df


# ---------------------- VENTANAS DESLIZANTES ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: Optional[List[str]] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Genera ventanas deslizantes de entrada-salida.

    - Si model_type es 'rf' o 'xgb': aplana las ventanas y devuelve DataFrames (2D).
    - Si model_type es 'rnn' o 'lstm': devuelve arrays 3D (muestras, tiempo, características).

    features: lista de nombres de columnas a usar como predictores.
              Si es None, se toman todas las columnas numéricas del dataframe.
    """
    if features is None:
        features = df.select_dtypes(include=[np.number]).columns.tolist()
        if target_col in features:
            features.remove(target_col)

    features = [f for f in features if f in df.columns]

    X_list, y_list, idxs = [], [], []
    total_len = len(df)

    for start in range(total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours

        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values

        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue

        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        X_flat = np.array(X_list)                     # (n, t, f)
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)                       # (n, output_hours)

        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d, index=idxs)
        y_df = pd.DataFrame(y_2d, columns=y_cols, index=idxs)
        return X_df, y_df
    else:
        return np.array(X_list), np.array(y_list)


# ---------------------- FUNCIÓN PRINCIPAL ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    """
    Procesa todos los archivos, guarda datasets individuales y combinados.
    RF y XGB se guardan en formato Parquet (comprimido con snappy).
    RNN/LSTM se guardan en formato .npz.
    """
    all_rf_X = []
    all_rf_y = []
    all_xgb_X = []
    all_xgb_y = []
    all_rnn_X = []
    all_rnn_y = []

    for p in file_paths:
        p = str(Path(p).expanduser())
        station_name = Path(p).stem
        print(f"\nProcesando estación: {station_name}")

        df = preprocess_single_csv(p)

        # ------------------------------------------------------------
        # 1. Datasets para RF y XGB (con one-hot encoding)
        # ------------------------------------------------------------
        cat_cols = [c for c in CATEGORICAL if c in df.columns]
        df_tab = pd.get_dummies(df, columns=cat_cols)

        feature_cols_tab = df_tab.select_dtypes(include=[np.number]).columns.tolist()
        if TARGET in feature_cols_tab:
            feature_cols_tab.remove(TARGET)

        X_rf, y_rf = make_sliding_windows(df_tab,
                                          features=feature_cols_tab,
                                          target_col=TARGET,
                                          model_type='rf')

        station_dir = output_dir / 'per_station' / station_name
        station_dir.mkdir(parents=True, exist_ok=True)

        # Guardar en Parquet (con compresión snappy)
        X_rf.to_parquet(station_dir / 'rf_X.parquet', compression='snappy', index=True)
        y_rf.to_parquet(station_dir / 'rf_y.parquet', compression='snappy', index=True)

        all_rf_X.append(X_rf)
        all_rf_y.append(y_rf)

        X_xgb, y_xgb = make_sliding_windows(df_tab,
                                            features=feature_cols_tab,
                                            target_col=TARGET,
                                            model_type='xgb')
        X_xgb.to_parquet(station_dir / 'xgb_X.parquet', compression='snappy', index=True)
        y_xgb.to_parquet(station_dir / 'xgb_y.parquet', compression='snappy', index=True)
        all_xgb_X.append(X_xgb)
        all_xgb_y.append(y_xgb)

        # ------------------------------------------------------------
        # 2. Datasets para RNN/LSTM (sin one-hot)
        # ------------------------------------------------------------
        feature_cols_rnn = df.select_dtypes(include=[np.number]).columns.tolist()
        if TARGET in feature_cols_rnn:
            feature_cols_rnn.remove(TARGET)

        X_rnn, y_rnn = make_sliding_windows(df,
                                            features=feature_cols_rnn,
                                            target_col=TARGET,
                                            model_type='rnn')
        np.savez_compressed(station_dir / 'rnn_lstm_data.npz', X=X_rnn, y=y_rnn)
        all_rnn_X.append(X_rnn)
        all_rnn_y.append(y_rnn)

        print(f"  Ventanas generadas: RF={len(X_rf)}, XGB={len(X_xgb)}, RNN={len(X_rnn)}")

    # ------------------------------------------------------------
    # 3. Datasets combinados
    # ------------------------------------------------------------
    print("\nGenerando datasets combinados...")
    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    if all_rf_X:
        combined_rf_X = pd.concat(all_rf_X, axis=0)
        combined_rf_y = pd.concat(all_rf_y, axis=0)
        combined_rf_X.to_parquet(combined_dir / 'rf_X.parquet', compression='snappy', index=True)
        combined_rf_y.to_parquet(combined_dir / 'rf_y.parquet', compression='snappy', index=True)

    if all_xgb_X:
        combined_xgb_X = pd.concat(all_xgb_X, axis=0)
        combined_xgb_y = pd.concat(all_xgb_y, axis=0)
        combined_xgb_X.to_parquet(combined_dir / 'xgb_X.parquet', compression='snappy', index=True)
        combined_xgb_y.to_parquet(combined_dir / 'xgb_y.parquet', compression='snappy', index=True)

    if all_rnn_X:
        combined_rnn_X = np.concatenate(all_rnn_X, axis=0)
        combined_rnn_y = np.concatenate(all_rnn_y, axis=0)
        np.savez_compressed(combined_dir / 'rnn_lstm_data.npz',
                            X=combined_rnn_X, y=combined_rnn_y)

    print(f"\n¡Proceso completado! Datos guardados en: {output_dir}")
    print("Los archivos de RF y XGB están en formato Parquet (comprimido).")


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)


Procesando estación: T1_E1_Alicante
  Preprocesando T1_E1_Alicante.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T1_E2_Elda
  Preprocesando T1_E2_Elda.csv...
  Ventanas generadas: RF=140113, XGB=140113, RNN=140113

Procesando estación: T2_E1_Elche
  Preprocesando T2_E1_Elche.csv...


OSError: [Errno 28] Error writing bytes to file. Detail: [errno 28] No space left on device

In [ ]:
"""
Script para entrenar y evaluar los 4 modelos (RF, XGB, RNN, LSTM)
utilizando los datasets generados por 'generar_datasets.py' (ahora en formato Parquet).

Se asume que los datos se encuentran en la estructura de carpetas:
  ~/Documents/GitHub/TFGFinal/Datos TFG/DatasetsFinales/
    per_station/
        <nombre_estacion>/
            rf_X.parquet, rf_y.parquet, xgb_X.parquet, xgb_y.parquet, rnn_lstm_data.npz
    combined/
            rf_X.parquet, rf_y.parquet, xgb_X.parquet, xgb_y.parquet, rnn_lstm_data.npz

El script permite elegir:
  - Modo 'station': procesa una estación concreta (por nombre).
  - Modo 'combined': procesa el dataset combinado.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, Optional, Union

# Modelos de sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# XGBoost
import xgboost as xgb

# TensorFlow / Keras para RNN y LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Configuración de rutas
BASE_DIR = Path(r"~/Documents/GitHub/TFGFinal/Datos TFG/DatasetsFinales").expanduser()
OUTPUT_DIR = BASE_DIR / "resultados_entrenamiento"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parámetros globales
INPUT_HOURS = 72
OUTPUT_HOURS = 72
RANDOM_STATE = 42
TEST_SIZE = 0.15          # 15% para test
VAL_SIZE = 0.15           # 15% para validación (sobre el resto después de test)
SCALE_FEATURES = True      # Escalar características (importante para RNN/LSTM y puede ayudar a RF/XGB)

# Selección de modo y estación (cambiar según necesidad)
MODE = 'combined'          # 'station' o 'combined'
STATION_NAME = 'T1_E1_Alicante'   # Solo si MODE == 'station'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def load_rf_xgb_data(mode: str, station: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Carga los datos de RF o XGB desde archivos Parquet.
    Restaura el índice datetime que se guardó como columna.
    """
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    X = pd.read_parquet(data_dir / 'rf_X.parquet')
    y = pd.read_parquet(data_dir / 'rf_y.parquet')

    # El índice se guardó como columna (por defecto se llama 'index' o la primera columna sin nombre)
    # Identificamos la columna que contiene las fechas (será de tipo datetime)
    for col in X.columns:
        if pd.api.types.is_datetime64_any_dtype(X[col]):
            X = X.set_index(col)
            break
    else:
        # Si no hay columna datetime, intentamos con la primera columna si parece una fecha
        first_col = X.columns[0]
        try:
            X[first_col] = pd.to_datetime(X[first_col])
            X = X.set_index(first_col)
        except:
            pass  # dejamos el índice numérico por defecto

    for col in y.columns:
        if pd.api.types.is_datetime64_any_dtype(y[col]):
            y = y.set_index(col)
            break
    else:
        first_col = y.columns[0]
        try:
            y[first_col] = pd.to_datetime(y[first_col])
            y = y.set_index(first_col)
        except:
            pass

    # Asegurar que el índice es datetime
    if not isinstance(X.index, pd.DatetimeIndex):
        X.index = pd.to_datetime(X.index)
    if not isinstance(y.index, pd.DatetimeIndex):
        y.index = pd.to_datetime(y.index)

    return X, y

def load_rnn_lstm_data(mode: str, station: Optional[str] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Carga los datos de RNN/LSTM (formato npz con arrays 3D)."""
    if mode == 'combined':
        data_dir = BASE_DIR / 'combined'
    else:
        data_dir = BASE_DIR / 'per_station' / station

    data = np.load(data_dir / 'rnn_lstm_data.npz')
    return data['X'], data['y']

def temporal_train_val_test_split(X, y, val_size=0.15, test_size=0.15, random_state=None):
    """
    Divide los datos en train/val/test respetando el orden temporal.
    Asume que X e y están indexados por tiempo (para pandas DataFrame) o
    que las muestras están en orden cronológico (para arrays numpy).
    """
    n = len(X)
    test_cut = int(n * (1 - test_size))
    val_cut = int(test_cut * (1 - val_size))

    if isinstance(X, pd.DataFrame):
        # Para pandas, mantener índices
        X_train, y_train = X.iloc[:val_cut], y.iloc[:val_cut]
        X_val, y_val = X.iloc[val_cut:test_cut], y.iloc[val_cut:test_cut]
        X_test, y_test = X.iloc[test_cut:], y.iloc[test_cut:]
    else:
        # Para numpy arrays
        X_train, y_train = X[:val_cut], y[:val_cut]
        X_val, y_val = X[val_cut:test_cut], y[val_cut:test_cut]
        X_test, y_test = X[test_cut:], y[test_cut:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def scale_features(X_train, X_val, X_test, feature_names=None):
    """
    Escala las características usando StandardScaler.
    Devuelve los conjuntos escalados y el scaler ajustado.
    """
    scaler = StandardScaler()
    # Ajustar con entrenamiento
    if isinstance(X_train, pd.DataFrame):
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        # Convertir de vuelta a DataFrame si se desea (opcional)
        X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
        X_val_scaled = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
        X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
    else:
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

def evaluate_model(y_true, y_pred, model_name, dataset_type='test'):
    """Calcula métricas y las imprime. Devuelve diccionario con las métricas."""
    mae = mean_absolute_error(y_true, y_pred, multioutput='uniform_average')
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='uniform_average'))
    r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
    print(f"{model_name} - {dataset_type} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

def plot_predictions(y_true, y_pred, model_name, num_samples=5):
    """Dibuja las primeras num_samples predicciones vs reales (para una hora específica, p.ej. horizonte 24)."""
    fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))
    if num_samples == 1:
        axes = [axes]
    for i in range(num_samples):
        ax = axes[i]
        ax.plot(y_true[i], label='Real', marker='o', linestyle='-')
        ax.plot(y_pred[i], label='Predicho', marker='x', linestyle='--')
        ax.set_title(f'{model_name} - Muestra {i+1}')
        ax.set_xlabel('Horas predicción')
        ax.set_ylabel('O3')
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_predicciones_ejemplo.png')
    plt.show()

# ---------------------- ENTRENAMIENTO DE MODELOS ----------------------

def train_random_forest(X_train, y_train, X_val, y_val, X_test, y_test):
    """Entrena Random Forest multi-output y evalúa."""
    print("\n--- Entrenando Random Forest ---")
    model = RandomForestRegressor(n_estimators=100, max_depth=20,
                                  random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train.values, y_pred_train, "RF", "train")
    metrics_val = evaluate_model(y_val.values, y_pred_val, "RF", "val")
    metrics_test = evaluate_model(y_test.values, y_pred_test, "RF", "test")

    import joblib
    joblib.dump(model, OUTPUT_DIR / 'random_forest_model.pkl')
    print("Modelo RF guardado en:", OUTPUT_DIR / 'random_forest_model.pkl')

    plot_predictions(y_test.values[:5], y_pred_test[:5], "Random Forest")
    return model, metrics_test

def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test):
    """Entrena XGBoost multi-output y evalúa."""
    print("\n--- Entrenando XGBoost ---")
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100,
                             max_depth=6, learning_rate=0.1,
                             random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train.values, y_pred_train, "XGB", "train")
    metrics_val = evaluate_model(y_val.values, y_pred_val, "XGB", "val")
    metrics_test = evaluate_model(y_test.values, y_pred_test, "XGB", "test")

    import joblib
    joblib.dump(model, OUTPUT_DIR / 'xgboost_model.pkl')
    print("Modelo XGB guardado en:", OUTPUT_DIR / 'xgboost_model.pkl')

    plot_predictions(y_test.values[:5], y_pred_test[:5], "XGBoost")
    return model, metrics_test

def build_rnn_model(input_shape, output_steps):
    model = Sequential([
        SimpleRNN(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        SimpleRNN(64, return_sequences=False),
        Dropout(0.2),
        Dense(output_steps)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(input_shape, output_steps):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(output_steps)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def train_rnn_lstm(model_type, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\n--- Entrenando {model_type.upper()} ---")
    input_shape = (X_train.shape[1], X_train.shape[2])
    output_steps = y_train.shape[1]

    if model_type == 'rnn':
        model = build_rnn_model(input_shape, output_steps)
    else:
        model = build_lstm_model(input_shape, output_steps)

    model.summary()

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)

    metrics_train = evaluate_model(y_train, y_pred_train, model_type.upper(), "train")
    metrics_val = evaluate_model(y_val, y_pred_val, model_type.upper(), "val")
    metrics_test = evaluate_model(y_test, y_pred_test, model_type.upper(), "test")

    model.save(OUTPUT_DIR / f'{model_type}_model.h5')
    np.save(OUTPUT_DIR / f'{model_type}_history.npy', history.history)
    print(f"Modelo {model_type.upper()} guardado en:", OUTPUT_DIR / f'{model_type}_model.h5')

    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'{model_type.upper()} - Loss')
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history.history['mae'], label='train_mae')
    plt.plot(history.history['val_mae'], label='val_mae')
    plt.title(f'{model_type.upper()} - MAE')
    plt.legend()
    plt.savefig(OUTPUT_DIR / f'{model_type}_training_history.png')
    plt.show()

    plot_predictions(y_test[:5], y_pred_test[:5], model_type.upper())

    return model, metrics_test

# ---------------------- MAIN ----------------------

def main():
    print(f"Modo seleccionado: {MODE}")
    if MODE == 'station':
        print(f"Estación: {STATION_NAME}")

    # 1. Cargar datos para RF/XGB (desde Parquet)
    X_rf, y_rf = load_rf_xgb_data(MODE, STATION_NAME if MODE=='station' else None)
    print(f"Datos RF/XGB cargados: {X_rf.shape[0]} muestras, {X_rf.shape[1]} features (aplanadas)")

    # División temporal para RF/XGB
    (X_rf_train, y_rf_train), (X_rf_val, y_rf_val), (X_rf_test, y_rf_test) = \
        temporal_train_val_test_split(X_rf, y_rf, val_size=VAL_SIZE, test_size=TEST_SIZE)
    print(f"RF/XGB - Train: {len(X_rf_train)}, Val: {len(X_rf_val)}, Test: {len(X_rf_test)}")

    # Escalado opcional para RF/XGB
    if SCALE_FEATURES:
        X_rf_train, X_rf_val, X_rf_test, scaler_rf = scale_features(
            X_rf_train, X_rf_val, X_rf_test)
        import joblib
        joblib.dump(scaler_rf, OUTPUT_DIR / 'scaler_rf_xgb.pkl')
        print("Scaler para RF/XGB guardado.")

    # 2. Entrenar RF
    rf_model, rf_metrics = train_random_forest(
        X_rf_train, y_rf_train,
        X_rf_val, y_rf_val,
        X_rf_test, y_rf_test
    )

    # 3. Entrenar XGBoost
    xgb_model, xgb_metrics = train_xgboost(
        X_rf_train, y_rf_train,
        X_rf_val, y_rf_val,
        X_rf_test, y_rf_test
    )

    # 4. Cargar datos para RNN/LSTM (desde .npz)
    X_rnn, y_rnn = load_rnn_lstm_data(MODE, STATION_NAME if MODE=='station' else None)
    print(f"Datos RNN/LSTM cargados: {X_rnn.shape[0]} muestras, timesteps={X_rnn.shape[1]}, features={X_rnn.shape[2]}")

    (X_rnn_train, y_rnn_train), (X_rnn_val, y_rnn_val), (X_rnn_test, y_rnn_test) = \
        temporal_train_val_test_split(X_rnn, y_rnn, val_size=VAL_SIZE, test_size=TEST_SIZE)
    print(f"RNN/LSTM - Train: {len(X_rnn_train)}, Val: {len(X_rnn_val)}, Test: {len(X_rnn_test)}")

    # Escalado para RNN/LSTM
    if SCALE_FEATURES:
        original_shape = X_rnn_train.shape
        n_samples_train, n_timesteps, n_features = original_shape
        X_rnn_train_2d = X_rnn_train.reshape(-1, n_features)
        X_rnn_val_2d = X_rnn_val.reshape(-1, n_features)
        X_rnn_test_2d = X_rnn_test.reshape(-1, n_features)

        scaler_rnn = StandardScaler()
        X_rnn_train_2d = scaler_rnn.fit_transform(X_rnn_train_2d)
        X_rnn_val_2d = scaler_rnn.transform(X_rnn_val_2d)
        X_rnn_test_2d = scaler_rnn.transform(X_rnn_test_2d)

        X_rnn_train = X_rnn_train_2d.reshape(original_shape)
        X_rnn_val = X_rnn_val_2d.reshape(X_rnn_val.shape)
        X_rnn_test = X_rnn_test_2d.reshape(X_rnn_test.shape)

        import joblib
        joblib.dump(scaler_rnn, OUTPUT_DIR / 'scaler_rnn_lstm.pkl')
        print("Scaler para RNN/LSTM guardado.")

    # 5. Entrenar RNN
    rnn_model, rnn_metrics = train_rnn_lstm(
        'rnn',
        X_rnn_train, y_rnn_train,
        X_rnn_val, y_rnn_val,
        X_rnn_test, y_rnn_test
    )

    # 6. Entrenar LSTM
    lstm_model, lstm_metrics = train_rnn_lstm(
        'lstm',
        X_rnn_train, y_rnn_train,
        X_rnn_val, y_rnn_val,
        X_rnn_test, y_rnn_test
    )

    # 7. Comparativa final
    print("\n" + "="*50)
    print("RESUMEN DE MÉTRICAS EN TEST")
    print("="*50)
    comparativa = pd.DataFrame({
        'RF': rf_metrics,
        'XGB': xgb_metrics,
        'RNN': rnn_metrics,
        'LSTM': lstm_metrics
    }).T
    print(comparativa)
    comparativa.to_csv(OUTPUT_DIR / 'comparativa_metricas.csv')

    print(f"\nTodos los resultados guardados en: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()